<a href="https://colab.research.google.com/github/SudemZeynep/adversarial-robustness-cnn-vit/blob/main/02_pretrained_setup_ipynb_koy_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install timm
!wget -q https://s3.amazonaws.com/fast-ai-imageclas/imagenette2.tgz
!tar -xzf imagenette2.tgz
print("hazır")

hazır


In [2]:
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision, torchvision.transforms as T
import timm, numpy as np, matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

# Normalizasyonu modelin ilk katmanı yapan sarmalayıcı
class Normalized(nn.Module):
    def __init__(self, model, mean, std):
        super().__init__()
        self.model = model
        self.register_buffer("mean", torch.tensor(mean).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor(std).view(1, 3, 1, 1))
    def forward(self, x):                      # x: [0, 1] aralığında
        return self.model((x - self.mean) / self.std)

# ResNet-50 (CNN), ImageNet'te önceden eğitilmiş
resnet = torchvision.models.resnet50(weights="IMAGENET1K_V2")
resnet = Normalized(resnet, [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]).to(device).eval()

# ViT-B/16 (Transformer); dynamic_img_size=True farklı çözünürlükleri kabul etmesini sağlar
vit = timm.create_model("vit_base_patch16_224.augreg2_in21k_ft_in1k",
                        pretrained=True, dynamic_img_size=True)
vit = Normalized(vit, [0.5, 0.5, 0.5], [0.5, 0.5, 0.5]).to(device).eval()

models = {"ResNet-50": resnet, "ViT-B/16": vit}
print("modeller hazır")

cuda
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 161MB/s]


model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

modeller hazır


In [3]:
# Imagenette'in 10 sınıfının ImageNet'teki sınıf numaraları
IMAGENET_IDX = torch.tensor([0, 217, 482, 491, 497, 566, 569, 571, 574, 701])
classes = ["tench", "springer", "cassette player", "chain saw", "church",
           "French horn", "garbage truck", "gas pump", "golf ball", "parachute"]

def load_data(res, n_per_class=20, seed=0):
    tf = T.Compose([T.Resize(res), T.CenterCrop(res), T.ToTensor()])   # [0, 1] aralığı
    ds = torchvision.datasets.ImageFolder("imagenette2/val", transform=tf)
    g = torch.Generator().manual_seed(seed)
    targets = torch.tensor(ds.targets)
    idx = torch.cat([torch.where(targets == c)[0][torch.randperm((targets == c).sum(), generator=g)[:n_per_class]]
                     for c in range(10)])
    sub = torch.utils.data.Subset(ds, idx.tolist())
    return torch.utils.data.DataLoader(sub, batch_size=16, shuffle=False)

loader_224 = load_data(224)
loader_512 = load_data(512)
print("224:", len(loader_224.dataset), "resim |", "512:", len(loader_512.dataset), "resim")

224: 200 resim | 512: 200 resim


In [4]:
@torch.no_grad()
def accuracy(model, loader):
    correct = 0
    for x, y in loader:
        x, y = x.to(device), IMAGENET_IDX[y].to(device)   # 10 sınıfı ImageNet numarasına çevir
        correct += (model(x).argmax(1) == y).sum().item()
    return correct / len(loader.dataset)

for name, m in models.items():
    print(f"{name:10s}  224×224: {accuracy(m, loader_224):.3f}   512×512: {accuracy(m, loader_512):.3f}")

ResNet-50   224×224: 0.920   512×512: 0.820
ViT-B/16    224×224: 0.915   512×512: 0.860


In [5]:
def fgsm(model, x, y, eps):
    x_adv = x.clone().requires_grad_(True)
    loss = F.cross_entropy(model(x_adv), y)
    grad = torch.autograd.grad(loss, x_adv)[0]
    return (x + eps * grad.sign()).clamp(0, 1).detach()

def pgd(model, x, y, eps, alpha=2/255, steps=10):
    x_adv = (x + torch.empty_like(x).uniform_(-eps, eps)).clamp(0, 1)      # rastgele başlangıç
    for _ in range(steps):
        x_adv = x_adv.detach().requires_grad_(True)
        loss = F.cross_entropy(model(x_adv), y)
        grad = torch.autograd.grad(loss, x_adv)[0]
        x_adv = x_adv + alpha * grad.sign()
        x_adv = torch.min(torch.max(x_adv, x - eps), x + eps).clamp(0, 1)  # ε izdüşümü
    return x_adv.detach()

def mi_fgsm(model, x, y, eps, steps=10, mu=1.0):
    alpha = eps / steps
    g = torch.zeros_like(x)                                                 # momentum birikimi
    x_adv = x.clone()
    for _ in range(steps):
        x_adv = x_adv.detach().requires_grad_(True)
        loss = F.cross_entropy(model(x_adv), y)
        grad = torch.autograd.grad(loss, x_adv)[0]
        g = mu * g + grad / grad.abs().mean(dim=(1, 2, 3), keepdim=True)    # önceki yön + yeni gradyan
        x_adv = x_adv + alpha * g.sign()
        x_adv = torch.min(torch.max(x_adv, x - eps), x + eps).clamp(0, 1)
    return x_adv.detach()

def input_diversity(x, p=0.5):
    if torch.rand(1).item() > p:                                            # p olasılıkla uygula
        return x
    res = x.shape[-1]
    rnd = torch.randint(int(res * 0.9), res, (1,)).item()                   # rastgele küçült
    x_r = F.interpolate(x, size=(rnd, rnd), mode="nearest")
    pad = res - rnd
    left, top = torch.randint(0, pad + 1, (2,)).tolist()                    # rastgele dolgu
    return F.pad(x_r, (left, pad - left, top, pad - top), value=0)

def di_fgsm(model, x, y, eps, steps=10, p=0.5):
    alpha = eps / steps
    x_adv = x.clone()
    for _ in range(steps):
        x_adv = x_adv.detach().requires_grad_(True)
        loss = F.cross_entropy(model(input_diversity(x_adv, p)), y)         # dönüştürülmüş görüntüde gradyan
        grad = torch.autograd.grad(loss, x_adv)[0]
        x_adv = x_adv + alpha * grad.sign()
        x_adv = torch.min(torch.max(x_adv, x - eps), x + eps).clamp(0, 1)
    return x_adv.detach()

attacks = {"FGSM": fgsm, "PGD": pgd, "MI-FGSM": mi_fgsm, "DI-FGSM": di_fgsm}
print("saldırılar hazır")

saldırılar hazır


In [6]:
EPS = 8/255

def generate(attack, src_model, loader, eps=EPS):
    xs, ys = [], []
    for x, y in loader:
        x, y = x.to(device), IMAGENET_IDX[y].to(device)
        xs.append(attack(src_model, x, y, eps).cpu())
        ys.append(y.cpu())
    return torch.cat(xs), torch.cat(ys)

torch.manual_seed(0)
adv_224 = {}
for a_name, atk in attacks.items():
    for s_name, src in models.items():
        adv_224[(a_name, s_name)] = generate(atk, src, loader_224)
        print(f"üretildi: {a_name:8s} ← {s_name}")

üretildi: FGSM     ← ResNet-50
üretildi: FGSM     ← ViT-B/16
üretildi: PGD      ← ResNet-50
üretildi: PGD      ← ViT-B/16
üretildi: MI-FGSM  ← ResNet-50
üretildi: MI-FGSM  ← ViT-B/16
üretildi: DI-FGSM  ← ResNet-50
üretildi: DI-FGSM  ← ViT-B/16


In [7]:
import pandas as pd

@torch.no_grad()
def acc_tensor(model, X, Y, bs=16):
    correct = 0
    for i in range(0, len(X), bs):
        correct += (model(X[i:i+bs].to(device)).argmax(1) == Y[i:i+bs].to(device)).sum().item()
    return correct / len(X)

rows = []
for (a_name, s_name), (X, Y) in adv_224.items():
    for t_name, tgt in models.items():
        rows.append({"Saldırı": a_name, "Üretilen": s_name, "Test edilen": t_name,
                     "Doğruluk": acc_tensor(tgt, X, Y)})

df_224 = pd.DataFrame(rows)
table = df_224.pivot_table(index=["Saldırı", "Üretilen"], columns="Test edilen",
                           values="Doğruluk", sort=False)
print(table.round(3))

Test edilen        ResNet-50  ViT-B/16
Saldırı Üretilen                      
FGSM    ResNet-50      0.580     0.870
        ViT-B/16       0.815     0.485
PGD     ResNet-50      0.025     0.900
        ViT-B/16       0.810     0.005
MI-FGSM ResNet-50      0.000     0.855
        ViT-B/16       0.725     0.005
DI-FGSM ResNet-50      0.115     0.885
        ViT-B/16       0.780     0.085


In [8]:
loader_512_small = torch.utils.data.DataLoader(loader_512.dataset, batch_size=8, shuffle=False)

torch.manual_seed(0)
adv_512 = {}
for a_name, atk in attacks.items():
    for s_name, src in models.items():
        adv_512[(a_name, s_name)] = generate(atk, src, loader_512_small)
        print(f"üretildi (512): {a_name:8s} ← {s_name}")
        torch.cuda.empty_cache()

üretildi (512): FGSM     ← ResNet-50
üretildi (512): FGSM     ← ViT-B/16
üretildi (512): PGD      ← ResNet-50
üretildi (512): PGD      ← ViT-B/16
üretildi (512): MI-FGSM  ← ResNet-50
üretildi (512): MI-FGSM  ← ViT-B/16
üretildi (512): DI-FGSM  ← ResNet-50
üretildi (512): DI-FGSM  ← ViT-B/16


In [9]:
def to_224(X):
    return F.interpolate(X, size=(224, 224), mode="bilinear", antialias=True, align_corners=False)

# Referans: saldırısız 512 resimlerin 224'e küçültülmüş hâli
X_clean = torch.cat([x for x, _ in loader_512])
Y_clean = torch.cat([IMAGENET_IDX[y] for _, y in loader_512])

rows = []
for t_name, tgt in models.items():
    rows.append({"Saldırı": "Temiz", "Üretilen": "-", "Test edilen": t_name,
                 "512 (doğrudan)": acc_tensor(tgt, X_clean, Y_clean, bs=8),
                 "512 → 224": acc_tensor(tgt, to_224(X_clean), Y_clean)})

for (a_name, s_name), (X, Y) in adv_512.items():
    X_small = to_224(X)
    for t_name, tgt in models.items():
        rows.append({"Saldırı": a_name, "Üretilen": s_name, "Test edilen": t_name,
                     "512 (doğrudan)": acc_tensor(tgt, X, Y, bs=8),
                     "512 → 224": acc_tensor(tgt, X_small, Y)})

df_512 = pd.DataFrame(rows)
print(df_512.round(3).to_string(index=False))

Saldırı  Üretilen Test edilen  512 (doğrudan)  512 → 224
  Temiz         -   ResNet-50           0.820      0.910
  Temiz         -    ViT-B/16           0.860      0.905
   FGSM ResNet-50   ResNet-50           0.375      0.910
   FGSM ResNet-50    ViT-B/16           0.835      0.900
   FGSM  ViT-B/16   ResNet-50           0.710      0.850
   FGSM  ViT-B/16    ViT-B/16           0.405      0.835
    PGD ResNet-50   ResNet-50           0.000      0.900
    PGD ResNet-50    ViT-B/16           0.860      0.910
    PGD  ViT-B/16   ResNet-50           0.725      0.895
    PGD  ViT-B/16    ViT-B/16           0.000      0.875
MI-FGSM ResNet-50   ResNet-50           0.000      0.890
MI-FGSM ResNet-50    ViT-B/16           0.850      0.905
MI-FGSM  ViT-B/16   ResNet-50           0.675      0.860
MI-FGSM  ViT-B/16    ViT-B/16           0.000      0.830
DI-FGSM ResNet-50   ResNet-50           0.010      0.905
DI-FGSM ResNet-50    ViT-B/16           0.840      0.915
DI-FGSM  ViT-B/16   ResNet-50  